# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook is the reproducible spine of
the deployed paper (`docs/index.html`) — every number on the public page traces back to a cell
here. It pulls together ML-02 through ML-10 rather than redoing them.

## 1. Question

*The research question and the decision it supports.*

**Question:** among a client's pages that are still visible in search, which ones should a
content team review *first* this month — and can a learned model do that ranking better than a
transparent rule, on the same held-out pages?

**Decision it supports:** content ops teams have limited review capacity and a portfolio too
large to check page-by-page. This output is a ranked queue with reason codes — not an
autopilot — that tells a human reviewer where to look first, and why.

In [1]:
print('Question: which visible pages should a content team review first, and does a learned')
print('model out-rank a transparent baseline rule on the same held-out split?')

Question: which visible pages should a content team review first, and does a learned
model out-rank a transparent baseline rule on the same held-out split?


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

- **Source:** `FlyRank/internship-warehouse` on Hugging Face (gated release), plus the local
  anonymized starter slice (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32
  pseudonymous clients) used for the notebooks that needed direct execution.
- **Iteration month:** `month=2026-03` for the warehouse data contract (ML-04/ML-05); the
  local slice is a fixed 90-day trailing snapshot with a 30-day-vs-previous-30-day trend label.
- **Sealed test month:** June 2026 (`_sample`), reserved and untouched by anything in this repo.
- **Excluded, and why:**
  - Every GA4-derived field (`sessions`, `engaged_sessions`, `scroll_events`, etc.) — coverage
    is uneven and `ga4_data_available` is three-valued, so a blind include would silently
    conflate "not tracked" with "zero engagement."
  - `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d` — these ARE
    the label or its direct raw inputs. ML-04's leakage trap and ML-09's confirmation both show
    exactly what including them does to AUC (see Methodology).
  - `provider_used`, `model_used` — the data dictionary marks these explicitly "not a model
    feature": they describe how content was produced, not how it performs.
  - `client_hash_id` / `client_id` — pseudonymous grouping key only, used for the client-holdout
    split, never fed to a model as a feature.
  - No client names, domains, URLs, or raw queries appear anywhere in this repo.

In [2]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'{len(df):,} rows, {df["client_id"].nunique()} pseudonymous clients')
print(f'columns used as features: 26 (18 numeric incl. 4 log-transformed, 8 categorical)')
print(f'columns explicitly excluded: GA4 fields, trend_direction/trend_pct, provider/model_used, IDs')

30,000 rows, 32 pseudonymous clients
columns used as features: 26 (18 numeric incl. 4 log-transformed, 8 categorical)
columns explicitly excluded: GA4 fields, trend_direction/trend_pct, provider/model_used, IDs


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

- **Label:** `is_declining` = 1 if `trend_direction == 'down'` (30-day impressions fell >10%
  vs. the previous 30 days). Base rate: 54.2% of the full dataset.
- **Features:** 18 numeric (traffic volume — log-transformed for skew, position, CTR, age,
  freshness, engagement) + 8 categorical (content type, intent, competition level, and tier
  buckets), one-hot encoded. Full list and exclusion rationale in `w05_model.ipynb`.
- **Baseline:** a transparent rule (ML-04/ML-07) — flags pages that are stale (no update in
  90+ days), visible (100+ impressions/90d), ranking in a reachable position tier, and
  underperforming their position tier's weighted-CTR benchmark.
- **Model:** Logistic Regression, chosen over Random Forest and a depth-3 Decision Tree after
  head-to-head comparison — the extra flexibility of a forest didn't earn its complexity here
  (`w05_model.ipynb`, section 1 and 3).
- **Validation design:** split **grouped by `client_id`** (`GroupShuffleSplit`, 80/20, seed 42)
  — never row-level random — because rows from the same client share systematic patterns a
  random split would let the model memorize instead of learn.
- **Leakage checks:** the label-derived-column trap (adding `trend_pct`/`impressions_last_30d`
  and watching AUC collapse toward-perfect), a product-flag naming-pattern scan, a population-
  selection check, and a random-vs-grouped split comparison — all in `w06_validation_audit.ipynb`.

In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct',
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].fillna('unknown').astype(str)
for c in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']:
    df[f'log_{c}'] = np.log1p(df[c])
NUM_FINAL = [c for c in NUMERIC_FEATURES if c not in
             ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']] + \
            ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']

X = df[NUM_FINAL + CATEGORICAL_FEATURES]
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f'Grouped split: {len(train_idx):,} train / {len(test_idx):,} test rows, '
      f'{len(overlap)} clients overlapping (must be 0)')
print(f'Base rate -- full: {y.mean():.3f}, test split: {y.iloc[test_idx].mean():.3f}')

Grouped split: 23,837 train / 6,163 test rows, 0 clients overlapping (must be 0)
Base rate -- full: 0.542, test split: 0.511


In [4]:
# Leakage confession, reproduced here (full detail + product-flag scan in w06):
pre = ColumnTransformer([('num', StandardScaler(), NUM_FINAL),
                          ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)])
pipe_honest = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipe_honest.fit(X.iloc[train_idx], y.iloc[train_idx])
auc_honest = roc_auc_score(y.iloc[test_idx], pipe_honest.predict_proba(X.iloc[test_idx])[:, 1])

X_leak = df[NUM_FINAL + CATEGORICAL_FEATURES + ['trend_pct', 'impressions_last_30d']].copy()
X_leak['trend_pct'] = pd.to_numeric(X_leak['trend_pct'], errors='coerce').fillna(0)
X_leak['impressions_last_30d'] = pd.to_numeric(X_leak['impressions_last_30d'], errors='coerce').fillna(0)
pre_leak = ColumnTransformer([('num', StandardScaler(), NUM_FINAL + ['trend_pct', 'impressions_last_30d']),
                               ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)])
pipe_leak = Pipeline([('pre', pre_leak), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipe_leak.fit(X_leak.iloc[train_idx], y.iloc[train_idx])
auc_leak = roc_auc_score(y.iloc[test_idx], pipe_leak.predict_proba(X_leak.iloc[test_idx])[:, 1])

print(f'Honest AUC:  {auc_honest:.3f}')
print(f'Leaked AUC:  {auc_leak:.3f}  (+{auc_leak - auc_honest:.3f} the instant trend_pct/'
      f'impressions_last_30d enter the feature set -- confirmed removed)')

Honest AUC:  0.616
Leaked AUC:  0.999  (+0.383 the instant trend_pct/impressions_last_30d enter the feature set -- confirmed removed)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [5]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return float(np.asarray(y_true)[order[:k]].mean())

ctr_benchmark = {'top_3': 0.334128, 'page_1': 0.354760, 'striking': 0.255782,
                 'page_3_5': 0.142359, 'deep': 0.055415}
stale = (df['days_since_last_update'] >= 90).astype(int)
visible = (df['impressions_90d'] >= 100).astype(int)
position_ok = df['position_tier'].isin(['top_3', 'page_1', 'striking', 'page_3_5']).astype(int)
benchmark = df['position_tier'].map(ctr_benchmark).fillna(0)
ctr_gap = (benchmark - df['ctr']).clip(lower=0)
baseline_score_full = (stale * visible * position_ok * ctr_gap * np.log1p(df['impressions_90d'])).values

y_test = y.iloc[test_idx].values
model_scores = pipe_honest.predict_proba(X.iloc[test_idx])[:, 1]
baseline_scores = baseline_score_full[test_idx]

Ks = [20, 50, 100, 200]
rows = []
for name, scores in [('Logistic Regression (model)', model_scores), ('Rule baseline (Week 4)', baseline_scores)]:
    row = {'method': name, 'ROC-AUC': round(float(roc_auc_score(y_test, scores)), 3)}
    for k in Ks:
        row[f'precision@{k}'] = round(precision_at_k(y_test, scores, k), 3)
    rows.append(row)
results = pd.DataFrame(rows).set_index('method')
results.loc['base rate (reference)'] = [np.nan] + [round(float(y_test.mean()), 3)] * len(Ks)
results

**Headline result:** the model beats the baseline on every metric on the identical held-out,
client-grouped test split — ROC-AUC 0.616 vs. ~0.50, precision@20 0.70 vs. base rate 0.51. The
baseline isn't a bad rule — it targets a narrower, more specific opportunity (visible + stale +
underperforming CTR for its position) rather than "predict decline" in general — but on the
exact question this capstone asks, the learned model ranks better.

## 5. Limitations

*What this work cannot claim.*

In [6]:
print('- Cross-sectional, one 90-day local snapshot: this ranks pages by OBSERVED association')
print('  with decline; it does not claim refreshing a flagged page WILL fix it (no experiment).')
print('- 32 clients, one dataset: not validated across industries or client sizes outside this')
print('  slice; re-check before trusting the queue on a very different portfolio.')
print('- AUC 0.616 is meaningfully above chance, far from perfect -- expect real false')
print('  positives/negatives at these volumes (see w05, section 4).')
print('- The label itself has a known failure mode: at very low impression volume, a 1-2')
print('  impression swing can flip trend_direction, producing label noise the model cannot')
print('  distinguish from a genuine decline (w05, false-negative analysis).')
print('- ~51%% of the model-only-high-risk reason-code bucket sits at exactly 0%% CTR -- as')
print('  likely a tracking gap as a real content problem (w07). This queue is decision-support')
print('  for a human reviewer, never an autopilot.')

- Cross-sectional, one 90-day local snapshot: this ranks pages by OBSERVED association
  with decline; it does not claim refreshing a flagged page WILL fix it (no experiment).
- 32 clients, one dataset: not validated across industries or client sizes outside this
  slice; re-check before trusting the queue on a very different portfolio.
- AUC 0.616 is meaningfully above chance, far from perfect -- expect real false
  positives/negatives at these volumes (see w05, section 4).
- The label itself has a known failure mode: at very low impression volume, a 1-2
  impression swing can flip trend_direction, producing label noise the model cannot
  distinguish from a genuine decline (w05, false-negative analysis).
- ~51%% of the model-only-high-risk reason-code bucket sits at exactly 0%% CTR -- as
  likely a tracking gap as a real content problem (w07). This queue is decision-support
  for a human reviewer, never an autopilot.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [7]:
test_df = df.iloc[test_idx].copy()
test_df['model_proba'] = model_scores
test_df['baseline_score'] = baseline_scores
test_df['baseline_flagged'] = test_df['baseline_score'] > 0
thresh = np.quantile(model_scores, 0.90)
test_df['model_flagged'] = test_df['model_proba'] >= thresh
test_df['actual'] = y_test

def reason(r):
    if r['model_flagged'] and r['baseline_flagged']: return 'baseline_and_model_agree'
    if r['model_flagged']: return 'model_only_high_risk'
    if r['baseline_flagged']: return 'baseline_only_ctr_gap'
    return 'not_flagged'
test_df['reason_code'] = test_df.apply(reason, axis=1)

summary = (test_df[test_df['reason_code'] != 'not_flagged']
           .groupby('reason_code').agg(n=('content_id', 'count'), observed_decline_rate=('actual', 'mean')))
print('Ranked action queue -- reason codes, ordered by how much to trust them:')
print(summary.sort_values('observed_decline_rate', ascending=False).round(3))
print()
print('1. baseline_and_model_agree first -- two independent signals point the same way.')
print('2. model_only_high_risk second -- highest observed decline rate, but check 0% CTR first.')
print('3. baseline_only_ctr_gap last -- barely above base rate; lower priority.')

Ranked action queue -- reason codes, ordered by how much to trust them:
                            n  observed_decline_rate
reason_code                                         
model_only_high_risk      465                  0.677
baseline_and_model_agree  152                  0.612
baseline_only_ctr_gap     350                  0.514

1. baseline_and_model_agree first -- two independent signals point the same way.
2. model_only_high_risk second -- highest observed decline rate, but check 0% CTR first.
3. baseline_only_ctr_gap last -- barely above base rate; lower priority.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [8]:
import os
os.makedirs('../outputs', exist_ok=True)

results.to_csv('../outputs/capstone_results_table.csv')
summary.to_csv('../outputs/capstone_reason_code_summary.csv')

chart_data = {
    'results_table': results.reset_index().to_dict(orient='records'),
    'reason_code_summary': summary.reset_index().to_dict(orient='records'),
    'base_rate_test': float(y_test.mean()),
    'auc_honest': float(auc_honest),
    'auc_leak': float(auc_leak),
    'n_train': int(len(train_idx)),
    'n_test': int(len(test_idx)),
    'n_clients': int(df['client_id'].nunique()),
}
import json as _json
with open('../outputs/capstone_chart_data.json', 'w') as f:
    _json.dump(chart_data, f, indent=2)

print('Exported for the deployed paper:')
print('  work/outputs/capstone_results_table.csv')
print('  work/outputs/capstone_reason_code_summary.csv')
print('  work/outputs/capstone_chart_data.json  (feeds docs/index.html directly)')

Exported for the deployed paper:
  work/outputs/capstone_results_table.csv
  work/outputs/capstone_reason_code_summary.csv
  work/outputs/capstone_chart_data.json  (feeds docs/index.html directly)


## ML-12 — Repurpose the work

*5-minute demo outline, a social-post cut, and a 3-sentence employer-facing summary.*

**5-minute demo outline:**
1. (30s) The question: which visible pages should a content team check first, out of a
   portfolio too large to review by hand?
2. (60s) Show the baseline rule — simple, transparent, and why it's a fair comparison target.
3. (90s) Show the leakage trap live: add `trend_pct` back in, watch AUC jump from 0.616 to
   0.998 — the single most convincing 30 seconds of the whole project.
4. (90s) The honest results table: model vs. baseline, same split, base rate visible.
5. (60s) The ranked queue with reason codes, and the one thing a reviewer must check first
   (0% CTR pages) before trusting `model_only_high_risk`.

**Social-post cut:** "I built a content-refresh scoring model on 30K real search-performance
rows — and the first thing it taught me was how easily a model can cheat. Adding one
leaked column pushed AUC from 0.62 to 0.998. The honest version still beats a rule-based
baseline on a held-out, client-grouped split. Full write-up + reproducible notebooks: [link]."

**Employer-facing summary (3 sentences):** I built a content-decline scoring model on a
30,000-row, 32-client search-performance dataset, comparing a transparent rule baseline against
a Logistic Regression model on an honest, client-grouped holdout split. The model improved
ROC-AUC from ~0.50 to 0.616 and precision@20 from the 51% base rate to 70%, while a leakage
audit confirmed the gains were real rather than an artifact of label-derived features or a
leaky split (a naive split would have overstated AUC by +0.095). The output is a ranked,
reason-coded action queue with explicit human-review triggers — built for decision support,
not automation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
      **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post
      cut + a 3-sentence employer-facing summary.